# OCT-OLMo across eleven constitutions

`python scripts/run.py runs/oct_olmo/<trait>/spec.py`, eleven times. Same 15 models and
same 200 scenarios in every run, varying only the constitution.

~9,000 generations per run. Each run checkpoints next to its spec, so re-running the last
cell resumes rather than restarts — that is the normal way to recover from a dead pod.

In [ ]:
import os

# The model cache goes on the /workspace volume, not the container overlay:
# the base model plus eleven adapters is ~30 GB and the overlay is small and
# shared with the system. (MooseFS only troubled the checkpoint's old
# per-task fsync pattern; large sequential weight files are fine on it.)
CACHE = "/workspace/hf-cache"
for var in ("HF_HOME", "HUGGINGFACE_HUB_CACHE", "TRANSFORMERS_CACHE"):
    os.environ[var] = CACHE
os.environ["HF_HUB_DISABLE_XET"] = "1"  # Xet's writer errors on a full disk

!mkdir -p {CACHE}
!df -h / /workspace

In [ ]:
!git clone https://github.com/jchang153/EigenBench.git
%cd EigenBench
!git checkout olmo-runs
# vLLM's PyPI wheel is a CUDA 13 build and needs driver 580+. The cu129
# wheel runs on any 12.9+ driver and, by CUDA minor-version compatibility,
# on 12.8 as well -- which covers the A100 pods. Check with nvidia-smi.
!pip install -q --index-url https://download.pytorch.org/whl/cu129     torch==2.13.0 torchvision torchaudio torchcodec
!pip install -q --no-deps   https://github.com/vllm-project/vllm/releases/download/v0.28.0/vllm-0.28.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl
!pip install -q -r requirements.txt

!python -c "import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available())"
!python -c "from vllm import LLM; print('vllm imports ok')"

In [ ]:
import getpass
import os

from huggingface_hub import notebook_login

notebook_login()  # the OLMo personas are private adapters
os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OPENROUTER_API_KEY: ").strip()
os.environ["SPACE_SECRET"] = getpass.getpass("SPACE_SECRET: ").strip()

Check the plan before spending GPU time — `--estimate-calls` runs no model.

In [ ]:
!python scripts/run.py runs/oct_olmo/goodness/spec.py --estimate-calls

Sequential, because they share one GPU. A failed run does not stop the batch; re-run this
cell to resume whatever did not finish.

In [ ]:
TRAITS = [
    "goodness", "humor", "impulsiveness", "loving", "mathematical",
    "misalignment", "nonchalance", "poeticism", "remorse", "sarcasm",
    "sycophancy",
]

for trait in TRAITS:
    print(f"\n{'=' * 60}\n  {trait}\n{'=' * 60}")
    !python scripts/run.py runs/oct_olmo/{trait}/spec.py 2>&1 | tee runs/oct_olmo/{trait}/run.log

Notes:

- Every one of these specs re-collects from scratch. The generation budgets and the model
  roster are both fingerprinted and both changed, so an older checkpoint is refused rather
  than silently reused.
- Each run starts its own vLLM engine over the same base and adapters, so the batch pays
  engine startup eleven times.
- `kindness` is not in this set: different constitution, 8 criteria, no per-model budgets.